In [30]:
import json
import re

# JSON formatting functions
def convert_dataturks_to_spacy(dataturks_JSON_FilePath):
    training_data = []
    lines=[]
    with open(dataturks_JSON_FilePath, 'r', encoding='utf-8' )  as f:
        lines = f.readlines()

    for line in lines:
        data = json.loads(line)
        text = data['content'].replace("\n", " ")
        entities = []
        data_annotations = data['annotation']
        if data_annotations is not None:
            for annotation in data_annotations:
                #only a single point in text annotation.
                point = annotation['points'][0]
                labels = annotation['label']
                # handle both list of labels or a single label.
                if not isinstance(labels, list):
                    labels = [labels]

                for label in labels:
                    point_start = point['start']
                    point_end = point['end']
                    point_text = point['text']

                    lstrip_diff = len(point_text) - len(point_text.lstrip())
                    rstrip_diff = len(point_text) - len(point_text.rstrip())
                    if lstrip_diff != 0:
                        point_start = point_start + lstrip_diff
                    if rstrip_diff != 0:
                        point_end = point_end - rstrip_diff
                    entities.append((point_start, point_end + 1 , label))
        training_data.append((text, {"entities" : entities}))
    return training_data

def trim_entity_spans(data: list) -> list:

    invalid_span_tokens = re.compile(r'\s')

    cleaned_data = []
    for text, annotations in data:
        entities = annotations['entities']
        valid_entities = []
        for start, end, label in entities:
            valid_start = start
            valid_end = end
            while valid_start < len(text) and invalid_span_tokens.match(
                    text[valid_start]):
                valid_start += 1
            while valid_end > 1 and invalid_span_tokens.match(
                    text[valid_end - 1]):
                valid_end -= 1
            valid_entities.append([valid_start, valid_end, label])
        cleaned_data.append([text, {'entities': valid_entities}])
    return cleaned_data

In [2]:
data = trim_entity_spans(convert_dataturks_to_spacy("Entity Recognition in Resumes.json"))
data[1]

['Afreen Jamadar Active member of IIIT Committee in Third year  Sangli, Maharashtra - Email me on Indeed: indeed.com/r/Afreen-Jamadar/8baf379b705e37c6  I wish to use my knowledge, skills and conceptual understanding to create excellent team environments and work consistently achieving organization objectives believes in taking initiative and work to excellence in my work.  WORK EXPERIENCE  Active member of IIIT Committee in Third year  Cisco Networking -  Kanpur, Uttar Pradesh  organized by Techkriti IIT Kanpur and Azure Skynet. PERSONALLITY TRAITS: • Quick learning ability • hard working  EDUCATION  PG-DAC  CDAC ACTS  2017  Bachelor of Engg in Information Technology  Shivaji University Kolhapur -  Kolhapur, Maharashtra  2016  SKILLS  Database (Less than 1 year), HTML (Less than 1 year), Linux. (Less than 1 year), MICROSOFT ACCESS (Less than 1 year), MICROSOFT WINDOWS (Less than 1 year)  ADDITIONAL INFORMATION  TECHNICAL SKILLS:  • Programming Languages: C, C++, Java, .net, php. • Web 

In [3]:
 def clean_entities(training_data):

     clean_data = []
     for text, annotation in training_data:

         entities = annotation.get('entities')
         entities_copy = entities.copy()

         # append entity only if it is longer than its overlapping entity
         i = 0
         for entity in entities_copy:
             j = 0
             for overlapping_entity in entities_copy:
                 # Skip self
                 if i != j:
                     e_start, e_end, oe_start, oe_end = entity[0], entity[1], overlapping_entity[0], overlapping_entity[1]
                     # Delete any entity that overlaps, keep if longer
                     if ((e_start >= oe_start and e_start <= oe_end) \
                     or (e_end <= oe_end and e_end >= oe_start)) \
                     and ((e_end - e_start) <= (oe_end - oe_start)):
                         entities.remove(entity)
                 j += 1
             i += 1
         clean_data.append((text, {'entities': entities}))

     return clean_data

 data = clean_entities(data)

In [4]:
data =  clean_entities(data)

In [5]:
import random
import math

def train_test_split(data, test_size, random_state):

    random.Random(random_state).shuffle(data)
    test_idx = len(data) - math.floor(test_size * len(data))
    train_set = data[0: test_idx]
    test_set = data[test_idx: ]

    return train_set, test_set

In [6]:
train_data, test_data = train_test_split(data, test_size = 0.1, random_state = 42)

In [7]:
import spacy
import random
from spacy.training import Example

def train_spacy(train_data, n_iter=100):
    nlp = spacy.blank('en')
    ner = nlp.add_pipe('ner')

    # اضافه کردن لیبل‌ها
    for _, annotations in train_data:
        for ent in annotations.get("entities"):
            ner.add_label(ent[2])

    other_pipes = [pipe for pipe in nlp.pipe_names if pipe != 'ner']
    with nlp.disable_pipes(*other_pipes):
        optimizer = nlp.begin_training()
        for itn in range(n_iter):
            print(f"Iteration {itn}")
            random.shuffle(train_data)
            losses = {}
            for text, annotations in train_data:
                try:
                    doc = nlp.make_doc(text)
                    example = Example.from_dict(doc, annotations)
                    nlp.update([example], drop=0.2, sgd=optimizer, losses=losses)
                except Exception as e:
                    print(f"Skipped: {e}")
            print(" Loss:", losses)

    return nlp


In [8]:
nlp_model = train_spacy(train_data)

Iteration 0


C:\Users\ASUS\AppData\Roaming\Python\Python311\site-packages\spacy\training\iob_utils.py:149: UserWarning: [W030] Some entities could not be aligned in the text "Ananya Chavan lecturer - oracle tutorials  Mumbai,..." with entities "[[2010, 2013, 'Degree'], [973, 1703, 'Skills'], [9...". Use `spacy.training.offsets_to_biluo_tags(nlp.make_doc(text), entities)` to check the alignment. Misaligned entities ('-') will be ignored during training.
  warnings.warn(
C:\Users\ASUS\AppData\Roaming\Python\Python311\site-packages\spacy\training\iob_utils.py:149: UserWarning: [W030] Some entities could not be aligned in the text "Vineeth Vijayan "Store Executive" - Orange City Ho..." with entities "[[6994, 7348, 'Skills'], [6936, 6972, 'College Nam...". Use `spacy.training.offsets_to_biluo_tags(nlp.make_doc(text), entities)` to check the alignment. Misaligned entities ('-') will be ignored during training.
  warnings.warn(
C:\Users\ASUS\AppData\Roaming\Python\Python311\site-packages\spacy\training\io

 Loss: {'ner': np.float32(15563.705)}
Iteration 1
 Loss: {'ner': np.float32(5840.39)}
Iteration 2
 Loss: {'ner': np.float32(4056.238)}
Iteration 3
 Loss: {'ner': np.float32(3607.9817)}
Iteration 4
 Loss: {'ner': np.float32(3419.3367)}
Iteration 5
 Loss: {'ner': np.float32(3138.7805)}
Iteration 6
 Loss: {'ner': np.float32(2894.3335)}
Iteration 7
 Loss: {'ner': np.float32(2740.8064)}
Iteration 8
 Loss: {'ner': np.float32(2540.4312)}
Iteration 9
 Loss: {'ner': np.float32(2465.6045)}
Iteration 10
 Loss: {'ner': np.float32(2351.2424)}
Iteration 11
 Loss: {'ner': np.float32(2375.6526)}
Iteration 12
 Loss: {'ner': np.float32(2169.5532)}
Iteration 13
 Loss: {'ner': np.float32(2050.0886)}
Iteration 14
 Loss: {'ner': np.float32(2088.423)}
Iteration 15
 Loss: {'ner': np.float32(1935.937)}
Iteration 16
 Loss: {'ner': np.float32(1832.5986)}
Iteration 17
 Loss: {'ner': np.float32(1873.2727)}
Iteration 18
 Loss: {'ner': np.float32(1800.8226)}
Iteration 19
 Loss: {'ner': np.float32(1693.5637)}
Iterati

In [16]:
nlp_model.to_disk("ner_resume_model")


In [17]:
nlp_model = spacy.load("ner_resume_model")

In [18]:
sample_idx = 0

text, ann = test_data[sample_idx]
doc = nlp_model(text)

# نمایش متن
print("text :\n", text[:500], "...\n")  # فقط 500 کرکتر اول

# موجودیت ‌های واقعی (ground truth)
print(" ground truth:")
for start, end, label in ann['entities']:
    print(f"{label:<20} → {text[start:end]}")

# موجودیت ‌های پیش‌بینی‌شده توسط مدل
print("\n predicted")
for ent in doc.ents:
    print(f"{ent.label_.ljust(20)} → {ent.text}")  # برای مثال


text :
 Shiksha Bhatnagar chnadigarh - Email me on Indeed: indeed.com/r/Shiksha-Bhatnagar/70e68b28225ca499  WORK EXPERIENCE  online job in home  Microsoft and copy past -  Chandigarh, Chandigarh -  August 2016 to July 2017  i need a online job so that i can attend  my regular college and i want to earn money that's it a part time online job so that i can do it on my phone or laptop  EDUCATION  pass 12 in medical  chandigarh university -  Chandigarh, Chandigarh  September 2016 to August 2019  SKILLS  Mic ...

 ground truth:
Skills               → Microsoft office and java (Less than 1 year)
College Name         → chandigarh university
Degree               → pass 12 in medical
Companies worked at  → Microsoft
Designation          → online job in home
Email Address        → indeed.com/r/Shiksha-Bhatnagar/70e68b28225ca499
Location             → chnadigarh
Name                 → Shiksha Bhatnagar

 predicted
Name                 → Shiksha Bhatnagar
Location             → chnadigarh
Email Ad

In [25]:
from seqeval.metrics import classification_report as seqeval_classification_report

def evaluate_token_level(test_data, model):
    true_labels = []
    pred_labels = []

    for text, ann in test_data:
        doc = model(text)
        tokens = [token.text for token in doc]
        true_tags = ['O'] * len(tokens)
        pred_tags = ['O'] * len(tokens)

        for start, end, label in ann['entities']:
            span = doc.char_span(start, end, alignment_mode='expand')
            if span is not None:
                for i in range(span.start, span.end):
                    prefix = 'B' if i == span.start else 'I'
                    true_tags[i] = f"{prefix}-{label}"

        for ent in doc.ents:
            for i in range(ent.start, ent.end):
                prefix = 'B' if i == ent.start else 'I'
                pred_tags[i] = f"{prefix}-{ent.label_}"

        true_labels.append(true_tags)
        pred_labels.append(pred_tags)

    print("Token-level Evaluation (BIO):")
    print(seqeval_classification_report(true_labels, pred_labels, digits=3))

In [26]:
from collections import Counter
#ارزیابی مدل در سطج موجودیت
def evaluate_entity_level(test_data, model):
    correct = Counter()
    predicted = Counter()
    gold = Counter()

    for text, ann in test_data:
        true_ents = set([(start, end, label) for start, end, label in ann['entities']])
        doc = model(text)
        pred_ents = set([(ent.start_char, ent.end_char, ent.label_) for ent in doc.ents])

        for ent in true_ents:
            gold[ent[2]] += 1
        for ent in pred_ents:
            predicted[ent[2]] += 1
        for ent in true_ents & pred_ents:
            correct[ent[2]] += 1

    print("Entity-level Evaluation (Exact Match):")
    print(f"{'Entity':<20} {'Precision':<10} {'Recall':<10} {'F1-Score':<10}")
    for label in sorted(set(list(gold.keys()) + list(predicted.keys()))):
        p = correct[label] / predicted[label] if predicted[label] > 0 else 0
        r = correct[label] / gold[label] if gold[label] > 0 else 0
        f1 = 2 * p * r / (p + r) if p + r > 0 else 0
        print(f"{label:<20} {p:<10.3f} {r:<10.3f} {f1:<10.3f}")


In [29]:
evaluate_token_level(test_data, nlp_model)


Token-level Evaluation (BIO):
                     precision    recall  f1-score   support

       College Name      0.643     0.500     0.563        36
Companies worked at      0.578     0.361     0.444        72
             Degree      0.808     0.750     0.778        28
        Designation      0.513     0.417     0.460        48
      Email Address      0.667     0.667     0.667        21
    Graduation Year      0.261     0.273     0.267        22
           Location      0.630     0.486     0.548        35
               Name      0.955     0.913     0.933        23
             Skills      0.478     0.333     0.393        33
Years of Experience      0.000     0.000     0.000         5

          micro avg      0.604     0.477     0.533       323
          macro avg      0.553     0.470     0.505       323
       weighted avg      0.593     0.477     0.525       323



In [22]:
evaluate_entity_level(test_data, nlp_model)

Entity-level Evaluation (Exact Match):
Entity               Precision  Recall     F1-Score  
College Name         0.643      0.500      0.563     
Companies worked at  0.578      0.361      0.444     
Degree               0.808      0.750      0.778     
Designation          0.513      0.417      0.460     
Email Address        0.667      0.667      0.667     
Graduation Year      0.261      0.273      0.267     
Location             0.630      0.486      0.548     
Name                 0.955      0.913      0.933     
Skills               0.478      0.333      0.393     
Years of Experience  0.000      0.000      0.000     
